In [1]:
import torch
import torchaudio
import soundfile as sf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import subprocess
subprocess.run(["pip", "install", "pdfplumber", "-q"])
import pdfplumber
from pathlib import Path

BASE_DIR      = Path("../")
RAW_DIR       = BASE_DIR / "data/raw/archive"
AUDIO_DIR     = RAW_DIR / "files"
PDF_PATH      = RAW_DIR / "listofwords.pdf"
PROCESSED_DIR = BASE_DIR / "data/processed"
PROCESSED_DIR.mkdir(exist_ok=True)

print(f"✅ torch      : {torch.__version__}")
print(f"✅ torchaudio : {torchaudio.__version__}")
print(f"Audio dir exists : {AUDIO_DIR.exists()}")
print(f"PDF exists       : {PDF_PATH.exists()}")

✅ torch      : 2.10.0+cpu
✅ torchaudio : 2.10.0+cpu
Audio dir exists : True
PDF exists       : True


In [2]:
words = {}
with pdfplumber.open(PDF_PATH) as pdf:
    for page in pdf.pages:
        text = page.extract_text()
        if not text:
            continue
        for line in text.split('\n'):
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            if len(parts) >= 2:
                num_str = parts[-1]       # number at end in raw RTL
                word    = ' '.join(parts[:-1])   # word at start
                urdu_to_eng = str.maketrans('۰۱۲۳۴۵۶۷۸۹', '0123456789')
                num_str = num_str.translate(urdu_to_eng)
                try:
                    words[int(num_str)] = word
                except:
                    continue

print(f"✅ Total words extracted: {len(words)}")
for i in [1, 2, 3, 4, 5]:
    print(f"  Word {i:03d} → {words.get(i, 'N/A')}")

✅ Total words extracted: 205
  Word 001 → صفر
  Word 002 → ی ا
  Word 003 → ود
  Word 004 → تین
  Word 005 → رچا


In [3]:
records = []
for speaker_dir in sorted(AUDIO_DIR.iterdir()):
    if not speaker_dir.is_dir():
        continue
    speaker = speaker_dir.name
    gender  = 'Male' if speaker[2] == 'M' else 'Female'
    native  = 'Native' if speaker[3] == 'Y' else 'Non-Native'
    group   = speaker[4:]
    for wav_file in sorted(speaker_dir.glob("*.wav")):
        word_num = int(wav_file.stem[-3:])
        word     = words.get(word_num, '')
        records.append({
            'file_path' : str(wav_file),
            'speaker'   : speaker,
            'gender'    : gender,
            'native'    : native,
            'group'     : group,
            'word_num'  : word_num,
            'word'      : word
        })

df = pd.DataFrame(records)
print(f"✅ Total files   : {len(df)}")
print(f"   Speakers      : {df['speaker'].nunique()}")
print(f"   Unique words  : {df['word_num'].nunique()}")
print(df.head(5))

✅ Total files   : 2500
   Speakers      : 10
   Unique words  : 250
                                        file_path speaker gender      native  \
0  ..\data\raw\archive\files\AAMNG1\AAMNG1001.wav  AAMNG1   Male  Non-Native   
1  ..\data\raw\archive\files\AAMNG1\AAMNG1002.wav  AAMNG1   Male  Non-Native   
2  ..\data\raw\archive\files\AAMNG1\AAMNG1003.wav  AAMNG1   Male  Non-Native   
3  ..\data\raw\archive\files\AAMNG1\AAMNG1004.wav  AAMNG1   Male  Non-Native   
4  ..\data\raw\archive\files\AAMNG1\AAMNG1005.wav  AAMNG1   Male  Non-Native   

  group  word_num word  
0    G1         1  صفر  
1    G1         2  ی ا  
2    G1         3   ود  
3    G1         4  تین  
4    G1         5  رچا  


In [4]:
all_text = ' '.join(df['word'].dropna().tolist())
vocab    = sorted(set(all_text))
vocab    = ['<blank>'] + vocab

char2idx = {ch: i for i, ch in enumerate(vocab)}
idx2char = {i: ch for i, ch in enumerate(vocab)}

VOCAB_SIZE = len(vocab)
print(f"✅ Vocabulary size : {VOCAB_SIZE}")
print(f"   First 10       : {vocab[:10]}")
print(f"   blank index    : {char2idx['<blank>']}")

with open(PROCESSED_DIR / "vocab.json", "w", encoding="utf-8") as f:
    json.dump(char2idx, f, ensure_ascii=False, indent=2)
print(f"\n✅ Vocab saved to data/processed/vocab.json")

✅ Vocabulary size : 41
   First 10       : ['<blank>', ' ', 'ئ', 'ا', 'ب', 'ت', 'ث', 'ج', 'ح', 'خ']
   blank index    : 0

✅ Vocab saved to data/processed/vocab.json


In [5]:
def extract_mel(wav_path, sample_rate=16000, n_mels=80, n_fft=400, hop_length=160):
    audio_data, sr = sf.read(wav_path)

    if len(audio_data.shape) > 1:
        audio_data = audio_data.mean(axis=1)

    waveform = torch.tensor(audio_data, dtype=torch.float32).unsqueeze(0)

    if sr != sample_rate:
        waveform = torchaudio.transforms.Resample(sr, sample_rate)(waveform)

    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate=sample_rate, n_fft=n_fft,
        hop_length=hop_length, n_mels=n_mels
    )(waveform)
    mel_db = torchaudio.transforms.AmplitudeToDB()(mel)
    return mel_db.squeeze(0)   # (n_mels, time_frames)

# Test it
sample_path = df.iloc[0]['file_path']
feature     = extract_mel(sample_path)
print(f"✅ Feature shape : {feature.shape}  (n_mels, time_frames)")
print(f"   Min: {feature.min():.2f}  Max: {feature.max():.2f}")

✅ Feature shape : torch.Size([80, 61])  (n_mels, time_frames)
   Min: -52.33  Max: 26.59


In [6]:
def encode_text(word, char2idx):
    return [char2idx[ch] for ch in word if ch in char2idx]

def decode_text(indices, idx2char):
    return ''.join([idx2char[i] for i in indices if i != 0])

# Test it
sample_word = df.iloc[0]['word']
encoded     = encode_text(sample_word, char2idx)
decoded     = decode_text(encoded, idx2char)

print(f"✅ Original word : {sample_word}")
print(f"   Encoded       : {encoded}")
print(f"   Decoded back  : {decoded}")
print(f"   Match         : {sample_word == decoded}")

✅ Original word : صفر
   Encoded       : [16, 22, 12]
   Decoded back  : صفر
   Match         : True


In [7]:
from tqdm.notebook import tqdm

features_dir = PROCESSED_DIR / "features"
features_dir.mkdir(parents=True, exist_ok=True)

errors = []
for idx, row in tqdm(df.iterrows(), total=len(df)):
    try:
        feat      = extract_mel(row['file_path'])
        save_path = features_dir / f"{Path(row['file_path']).stem}.pt"
        torch.save(feat, save_path)
    except Exception as e:
        errors.append((row['file_path'], str(e)))

print(f"✅ Features saved : {len(df) - len(errors)}/{len(df)}")
print(f"   Errors         : {len(errors)}")

  0%|          | 0/2500 [00:00<?, ?it/s]

✅ Features saved : 2500/2500
   Errors         : 0


In [8]:
df.to_csv(PROCESSED_DIR / "metadata.csv", index=False, encoding='utf-8-sig')
print(f"✅ Metadata saved to data/processed/metadata.csv")
print(f"   Shape: {df.shape}")

✅ Metadata saved to data/processed/metadata.csv
   Shape: (2500, 7)
